In [1]:
import h5py
import numpy as np
import os
import matplotlib.pyplot as plt
import os
import tensorflow as tf

2026-03-16 09:19:37.957908: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773677977.972922  923028 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773677977.977867  923028 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773677977.991343  923028 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773677977.991355  923028 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773677977.991357  923028 computation_placer.cc:177] computation placer alr

In [4]:
# minorized reference
with h5py.File('/pscratch/sd/k/kberard/SCGSR/Data/vo2_1x1x1_opt/density_data/vmc_J2/density_tot_ref_mean.h5', 'r') as file:
    #print("Keys: %s" % file.keys())
    ref_d = file['density'][:]
#print(ref_d)
print(ref_d.shape)




(116, 116, 72)


In [5]:
def D_JS(p1,p2,tol=1e-16):
    p1=p1/np.sum(p1)
    p2=p2/np.sum(p2)
    pm = (p1+p2)/2
    p1_nonzero = np.abs(p1)>tol
    p2_nonzero = np.abs(p2)>tol
    p1  = np.abs(p1[p1_nonzero])
    pm1 = np.abs(pm[p1_nonzero])
    p2  = np.abs(p2[p2_nonzero])
    pm2 = np.abs(pm[p2_nonzero])
    d = .5*( (p1*np.log(p1/pm1)).sum() + (p2*np.log(p2/pm2)).sum() )
    d /= np.log(2) # normalize to max of 1
    return d

In [6]:
data = np.load("/pscratch/sd/k/kberard/SCGSR/EDDA/VO2/Data_Gen/DFT_Blobs/DFT_dat_lev_23922392000000_samples_Residual_Perturbed.npz")


In [7]:
print(data['x_train'].shape)

(1000, 116, 116, 72)


In [8]:
x_train = data['x_train']
x_val   = data['x_val']
y_train = data['y_train']
y_val   = data['y_val']
n_samples = data["total_samples"]

x_train = x_train[..., np.newaxis]
x_val   = x_val[..., np.newaxis]

y_train = y_train[..., np.newaxis]
y_val   = y_val[..., np.newaxis]
data = 0


In [9]:
dft_path ='/pscratch/sd/k/kberard/SCGSR/Data/vo2_1x1x1_opt/density_data/vmc_noJ/density_tot_ref.h5'
with h5py.File(dft_path, 'r') as file:
    dft_d = file['density'][:]
# 1. Get the residual map (what the network sees)
residual_map = x_train[0,:,:,:,0]
print(f"Residual Sum: {np.sum(residual_map):.2f} (Expected: close to 0, not 50)")

# 2. Inverse Transform to get Real Density
# Formula: Density = (Residual * sqrt(DFT)) + DFT
epsilon = 1e-9
dsqrt = np.sqrt(np.abs(dft_d) + epsilon)
restored_density = (residual_map * dsqrt) + dft_d

# 3. Check the Sum
total_electrons = np.sum(restored_density)
print(f"Total Electrons: {total_electrons:.4f} (Expected: ~50.0)")

Residual Sum: -4.11 (Expected: close to 0, not 50)
Total Electrons: 50.0063 (Expected: ~50.0)


In [10]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

def create_residual_unet(input_shape):
    inputs = layers.Input(shape=input_shape)
    
    # Standard U-Net (Depth 2 is fine for residuals)
    # Encoder
    c1 = layers.Conv3D(32, 3, padding='same', activation='relu')(inputs)
    c1 = layers.Conv3D(32, 3, padding='same', activation='relu')(c1)
    p1 = layers.MaxPooling3D(2)(c1)
    
    c2 = layers.Conv3D(64, 3, padding='same', activation='relu')(p1)
    c2 = layers.Conv3D(64, 3, padding='same', activation='relu')(c2)
    p2 = layers.MaxPooling3D(2)(c2)
    
    # Bottleneck
    b = layers.Conv3D(128, 3, padding='same', activation='relu')(p2)
    b = layers.Conv3D(128, 3, padding='same', activation='relu')(b)
    
    # Decoder
    u1 = layers.UpSampling3D(2)(b)
    u1 = layers.Concatenate()([u1, c2])
    d1 = layers.Conv3D(64, 3, padding='same', activation='relu')(u1)
    d1 = layers.Conv3D(64, 3, padding='same', activation='relu')(d1)
    
    u2 = layers.UpSampling3D(2)(d1)
    u2 = layers.Concatenate()([u2, c1])
    d2 = layers.Conv3D(32, 3, padding='same', activation='relu')(u2)
    d2 = layers.Conv3D(32, 3, padding='same', activation='relu')(d2)
    
    # Output: predicting the RESIDUAL, can be negative, so LINEAR activation
    outputs = layers.Conv3D(1, 1, padding='same', activation='linear')(d2)
    
    return models.Model(inputs, outputs)

# Compile & Train
model = create_residual_unet(x_train.shape[1:])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='mse') 

model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    batch_size=8,
    epochs=50,
    callbacks=[
        callbacks.EarlyStopping(patience=10, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(patience=5, factor=0.5)
    ]
)
model.save("residual_denoiser_40M_Blob.keras")

I0000 00:00:1773678019.965703  923028 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 38479 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:82:00.0, compute capability: 8.0


Epoch 1/50


I0000 00:00:1773678028.439300  923943 service.cc:152] XLA service 0x7f2b00114070 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773678028.439622  923943 service.cc:160]   StreamExecutor device (0): NVIDIA A100-SXM4-40GB, Compute Capability 8.0
2026-03-16 09:20:28.583154: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1773678028.959800  923943 cuda_dnn.cc:529] Loaded cuDNN version 90300
2026-03-16 09:20:31.198797: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng0{} for conv %cudnn-conv-bias-activation.34 = (f32[8,32,116,116,72]{4,3,2,1,0}, u8[0]{0}) custom-call(f32[8,32,116,116,72]{4,3,2,1,0} %bitcast.4987, f32[32,32,3,3,3]{4,3,2,1,0} %bitcast.4978, f32[32]{0} %bitcast.5037), window={size=3x3x3 pad=1_1x1_1x1_1}, dim_labels=bf012_oi012->bf012, custom_call_target="__cudnn$convBiasAct

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 283ms/step - loss: 5.9912e-08 

2026-03-16 09:22:13.522966: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng0{} for conv %cudnn-conv-bias-activation.34 = (f32[8,32,116,116,72]{4,3,2,1,0}, u8[0]{0}) custom-call(f32[8,32,116,116,72]{4,3,2,1,0} %bitcast.498, f32[32,32,3,3,3]{4,3,2,1,0} %bitcast.505, f32[32]{0} %bitcast.507), window={size=3x3x3 pad=1_1x1_1x1_1}, dim_labels=bf012_oi012->bf012, custom_call_target="__cudnn$convBiasActivationForward", metadata={op_type="Conv3D" op_name="functional_1/conv3d_1_2/convolution" source_file="/global/u2/k/kberard/environments/SCGSR/lib/python3.12/site-packages/tensorflow/python/framework/ops.py" source_line=1200}, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"conv_result_scale":1,"activation_mode":"kRelu","side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false} is taking a while...
2026-03-16 09:22:13.928094: E external/local_xla/xla/service/slow_operation_alarm.cc:140] The ope

125/125 ━━━━━━━━━━━━━━━━━━━━ 118s 398ms/step - loss: 5.9872e-08 - val_loss: 5.3409e-08 - learning_rate: 1.0000e-04
Epoch 2/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 37s 299ms/step - loss: 5.1211e-08 - val_loss: 4.7213e-08 - learning_rate: 1.0000e-04
Epoch 3/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 37s 299ms/step - loss: 4.3087e-08 - val_loss: 4.4087e-08 - learning_rate: 1.0000e-04
Epoch 4/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 37s 299ms/step - loss: 4.5699e-08 - val_loss: 4.0593e-08 - learning_rate: 1.0000e-04
Epoch 5/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 38s 299ms/step - loss: 3.6430e-08 - val_loss: 3.6280e-08 - learning_rate: 1.0000e-04
Epoch 6/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 38s 299ms/step - loss: 3.6483e-08 - val_loss: 3.1714e-08 - learning_rate: 1.0000e-04
Epoch 7/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 37s 299ms/step - loss: 2.8889e-08 - val_loss: 2.9270e-08 - learning_rate: 5.0000e-05
Epoch 8/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 38s 300ms/step - loss: 2.7204e-08 - val_loss: 2.6895e-08 - learning_rate: 5.0000e-05
Epoch 9/50